___

In [ ]:
# Imports
import pandas as pd

___
**_Bitte beachten:_**

- In Code-Zellen soll (bei Bedarf kommentierter) Code eingefügt werden. In Raw-Zellen erwarten wir Antworten im Freitext-Format.
- Fragestellungen sind teilweise bewusst etwas offener formuliert, weil Sie auch im Arbeitsleben nur selten mit ganz spezifischen Anweisungen rechnen können.Es gibt häufig mehrere mögliche Lösungen, dies wird in der Korrektur  berücksichtigt.

___

# (I can't get no) satisfaction
Wir arbeiten weiterhin mit dem Datensatz über die Zufriedenheit von Angestellten der fiktiven Firma *Fiktiva*. In dieser Kurseinheit werden wir einige Techniken zur Datenvorverarbeitung / zum Data Cleaning anwenden.

Sie können hier, wenn Sie weitere Informationen benötigen, ebenfalls externe Quellen verwenden und bspw. die Datei `metadata.txt` zu Rate ziehen, aber auch alle Informationen aus dem Fiktiva-Newsletter. 

In [ ]:
# Datensatz laden
# Laden Sie den Datensatz erneut so, dass die erste Spalte als Index (Row Label) verwendet wird
df = 

## Spalten umbennen

Die Spalte `Dept` wird als einzige groß geschrieben. Benennen Sie die Spalte deshalb in `department` um.

In [ ]:
# Ihre Lösung

## Spalten transformieren
Um ein besseres Gefühl für die Werte der Spalte `salary` zu bekommen, wandeln wir die Werte, die aktuell in US-Dollar angegeben sind, in Euro um. Aktualisieren Sie anschließend auch die Datei `metadata.txt`.

*Hinweis: Verwenden Sie folgenden Umrechnungskurs: $ 1\$ = 0.9€ $*

In [ ]:
# Ihre Lösung

Die Spalte `entry_date` enthält Informationen über das Einstellungsdatum, die Daten sind jedoch nicht vom Typ Datetime. Dies ist für Datums-Berechnungen sehr ungünstig, wandeln Sie den Datentypen deshalb entsprechend um.

In [ ]:
# Ihre Lösung

In der Spalte `gender` sind die Werte nicht einheitlich kodiert (`m`/`Male` und `f`/`Female`). Vereinheitlichen Sie diese Spalte, so dass `m` zu `Male` wird und `f` zu `Female`. Dazu können Sie untenstehendes Dictionary `gender_recoding` verwenden.

In [ ]:
gender_recoding = {
    "f": "Female",
    "m": "Male"
}

In [ ]:
# Ihre Lösung

In der Spalte `education` ist der Ausbildungsgrad der Angestellten in `UG`/`PG` angegeben:

In [ ]:
df["education"].unique()

Dies sind in Deutschland keine gebräuchlichen Abkürzungen, weshalb die Information u.U. nicht verständlich ist. Transformieren Sie die Spalte, so dass die Abkürzungen ausgeschrieben und somit verständlicher sind. Aktualisieren Sie außerdem die Datei `metadata.txt` und fügen Sie eine kurze Erklärung zu den beiden Ausprägungen hinzu.

In [ ]:
# Ihre Lösung

## Plausibilitätschecks
Es ist sinnvoll, einige Prüfungen bzgl. der Plausibilität der Daten vorzunhemen.

Nehmen Sie an, einer Ihrer Plausibilitätschecks schlägt fehlt. Was sollten Sie dann tun?

Wir würden davon ausgehen, dass Personen frühestens ab 16 Jahren fest in einer Firma angestellt werden. Prüfen Sie deshalb, ob alle Personen mindestens 16 Jahre alt waren, als Sie bei der Firma begonnen haben (siehe `entry_date`). Gehen Sie dafür davon aus, dass der Datensatz am 01.01.2021 erstellt wurde (siehe Variable `created`).

In [ ]:
from datetime import date
created = date(2021, 1, 1)

In [ ]:
# Ihre Lösung

Die ID der Angestellten scheint mit ihrer Abteilung zusammenzuhängen. Betrachten wir bspw. den ersten Datensatz:

In [ ]:
df.loc[1][["emp_id", "department"]]

Die ersten Buchstaben beziehen sich auf die Abteilung. Wir gruppieren das Dataframe und schauen uns aus jeder Abteilung die `emp_id` des ersten Datensatzes an.

In [ ]:
df.groupby("department").first()["emp_id"]

Wir gehen deshalb von folgender Zuordnung aus:

In [ ]:
id_dept_mapping = {
    "HR":"HR",
    "MKT":"Marketing",
    "PUR":"Purchasing",
    "SAL":"Sales",
    "TECH":"Technology"

}

Den Buchstaben folgt dann immer eine eindeutige Nummer, der Aufbau ist also \<dept_id\>\<employee_number\>.

Wir wollen prüfen, ob dieses Muster für alle Angestellten konsistent ist. Dafür verwenden wir die Funktion `check_id`. Diese prüft mittels des oben angegebenen Dictionary `id_dept_mapping`, ob `dept_id` und die Abteilung zusammenpassen. Zur Extraktion der `dept_id` sollen Sie einen regulären Ausdruck erstellen. Prüfen Sie anschließend mithilfe der Funktion `check_id`, ob das Muster für alle Angestellten konsistent ist.

*Hinweis: Sie können vom obigen Aufbau ausgehen, bei dem `dept_id` immer aus Buchstaben besteht, während `employee_number` immer Zahlen sein sollen.*

In [ ]:
import re
# Ihre Lösung
regular_expression = "" # <- fügen Sie hier den entsprechenden regulären Ausdruck ein

In [ ]:
def check_id(emp_id, department):
    dept_id = "".join(re.findall(regular_expression, emp_id))
    return id_dept_mapping.get(dept_id) == department

In [ ]:
# Ist der Check für alle Zeilen des Datensatzes erfolgreich?

## Fehlende Werte
Oft kommt es vor, dass in Daten Werte fehlen. Dies kann unterschiedlichste Gründe haben, z.B. kann ein Wert bewusst weggelassen werden (bpsw. der Titel in einer Anrede) oder Daten können nicht korrekt übertragen werden. Für viele Analysen ist es jedoch wichtig, dass es keine fehlenden Werte gibt.

Das Ersetzen von fehlenden Werten wird als *(Missing Value) Imputation* bezeichnet. Hierfür gibt es verschiedene Strategien:
- Korrekten Wert ermitteln: Im besten Fall kann der korrekte Wert ermittelt werden, bspw. durch Rücksprache mit (Fach-)Expert\*innen. Dies ist jedoch nur selten der Fall und häufig mit großem Aufwand verbunden.
- Zeilen mit fehlenden Werten löschen: Fehlen wenige, einzelne Werte, kann es u.U. sinnvoll sein, die entsprechenden Zeilen zu löschen. Hierbei muss jedoch darauf geachtet werden, dass nicht zu viele Informationen verloren gehen.
- Spalte mit fehlenden Werten löschen: Fehlen in einer Spalte so viele Werte, dass diese keine Aussagekraft mehr hat, kann es sinnvoll sein, die gesamte Spalte zu entfernen.
- Neuen Wert einsetzen:
    - Neue Kategorie: Bei nominalen Werten kann eine neue Kategorie (z.B. "Unbekannt") eingeführt werden.
    - Mode Imputation: Die fehlenden Werte werden durch den am häufigst vorkommenden Wert ersetzt.
    - Mean Imputation: Die fehlenden Werte werden durch den Mittelwert ersetzt.
    - Median Imputation: Die fehlenden Werte werden durch den Median ersetzt.
    - Mode, Mean und Median Imputation können auch basierend auf den Werten einer anderen Spalte vorgenommen werden (in unserem Fall z.B. für zufriedene und unzufriedene Angestellte getrennt).
    - Imputation mit 0 oder einem anderen konstanten Wert: Die fehlenden Werte werden durch eine 0 oder eine andere, je nach Anwendungsfall passende, Konstante ersetzt.
    - Modell-basierte Imputation: Es wird ein Machine-Learning-Modell trainiert, das die fehlenden Werte vorhersagt.
    - Zusätzlich zu den genannten Strategien ist es möglich, ein neues Feature zu erstellen, das kennzeichnet, ob eine Imputation vorgenommen wurde (=1) oder nicht (=0).

Wir bleiben bei der Spalte `gender`, hier gibt es fehlende Werte.

In [ ]:
df["gender"].isna().sum()

Wählen Sie eine geeignete Strategie zur *Imputation*, also der Ersetzung der fehlenden Werte. Begründen Sie Ihre Entscheidung!

In [ ]:
# Ihre Lösung

Auch in der Spalte `last_raise` gibt es fehlende Werte.

In [ ]:
df["last_raise"].isna().sum()

Entscheiden Sie sich hier ebenfalls für eine geeignete Imputation-Strategie und begründen Sie Ihre Wahl.

In [ ]:
# Ihre Analysen

In [ ]:
# Ihre Lösung

Eine weitere Spalte mit fehlenden Werten ist `rating`.

In [ ]:
df["rating"].isna().sum()

Entscheiden Sie sich hier ebenfalls für eine geeignete Imputation-Strategie und begründen Sie Ihre Wahl.

In [ ]:
# Ihre Analysen

In [ ]:
# Ihre Lösung

## Ausreißer

Betrachten wir die Histogramme für die Spalten `awards` und `certifications`:

In [ ]:
for col in ["awards", "certifications"]:
    df.hist(col)

In diesen beiden Spalten scheint es Ausreißer in Werten zu geben.

Betrachten wir zunächst die Spalte `awards`. Identifizieren Sie die Zeile, in der der Ausreißer vorkommt, und überlegen Sie sich eine geeignete Strategie, um mit diesem Wert umzugehen.

In [ ]:
# Ihre Analysen

In [ ]:
# Ihre Lösung

Wenden wir uns nun der Spalte `certifications` zu. Identifizieren Sie auch hier den Ausreißer und überlegen Sie sich eine geeignete Strategie, um mit diesem Wert umzugehen.

In [ ]:
# Ihre Analysen

In [ ]:
# Ihre Lösung

## Spalten/Zeilen löschen

Betrachten wir die Spalte `emp_id`. Aus wie vielen einzigartigen Werten besteht diese?

In [ ]:
# Ihre Lösung

Scheinbar gibt es eine ID doppelt. Schauen wir uns zunächst die Zeilen an, die die gleiche `emp_id` besitzen. Identifizieren Sie hierzu, um welche ID es sich handelt und lassen Sie sich dann die zugehörigen Zeilen ausgeben.

In [ ]:
# Ihre Lösung

Wählen Sie eine geeignete Strategie, um mit diesem Duplikat umzugehen. Begründen Sie Ihre Entscheidung.

In [ ]:
# Ihre Lösung

Betrachten wir nun die Werte der Spalte `emp_id`. Bringen diese Informationen einen Mehrwert zu der Vorhersage der Zufriedenheit? Wenn ja, bleibt die Spalte im Datensatz, wenn nein, transformieren Sie den Datensatz entsprechend.

In [ ]:
# Ihre Lösung

## Daten speichern

In [ ]:
df

Wenn, Sie alle Aufgaben korrekt ausgeführt haben, ist die Vorverarbeitung nun abgeschlossen:
- Alle Spalten liegen im passenden Datenformat vor
- Kategorielle Spalten wurden vereinheitlicht
- Es gibt keine Duplikate
- Es gibt keine fehlenden Werte
- Die Plausibilitätschecks wurden bestanden

Speichern Sie den von Ihnen transformierten Datensatz unter dem Namen `employees_satisfaction_transformed.csv` ab.

In [ ]:
# Ihre Lösung